In [ ]:
import json
import numpy as np

def _from_jsonable(obj):
    """Recursively convert JSON types back to numpy/complex objects.
    
    ``{"real": ..., "imag": ...}`` dicts become complex numbers;
    lists become numpy arrays if they contain numeric/complex values.
    """
    if isinstance(obj, dict):
        # Complex number encoding
        if set(obj.keys()) == {"real", "imag"}:
            return complex(obj["real"], obj["imag"])
        # Regular dict: recurse on values
        return {k: _from_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        converted = [_from_jsonable(v) for v in obj]
        # Convert to numpy array if all elements are numeric or complex
        if all(isinstance(v, (int, float, complex, np.ndarray)) for v in converted):
            return np.array(converted)
        return converted
    return obj  # int, float, str → keep as-is


def load_json(path):
    """Load a JSON file and return a dict with numpy/complex values."""
    with open(path, "r") as f:
        raw = json.load(f)
    return _from_jsonable(raw)

In [ ]:
data = load_json("/Users/tmalas/Desktop/Dev/Simulations_local/Cheb_Ar/output/cheb_ar_vs_epsp_alphasq_6.json")[0]

rate_bf = data["rate_bf"]        # complex scalar
x_ritz  = data["x_ritz"]        # np.ndarray of complex
params  = data["params"]         # dict of floats
mu_RQ   = data["res"]["mu_RQ"]  # complex scalar
res_rel = data["res"]["res_rel"] # float

In [ ]:
import matplotlib.pyplot as plt
import dynamiqs as dq
dq.set_precision("double")
n_a=20
n_b=11
alpha_sq=6 
rho_ritz = dq.unvectorize(dq.asqarray(x_ritz[:, None]))
rho_ritz = dq.asqarray(dq.to_jax(rho_ritz), dims=(n_a, n_b))
theta = np.linspace(0,2*np.pi, 100)

fig ,ax = plt.subplots()
dq.plot.wigner(dq.ptrace(rho_ritz,0), ax = ax)
ax.plot(np.sqrt(alpha_sq/2) * np.cos(theta), np.sqrt(alpha_sq/2) * np.sin(theta), color='r', 
        linewidth = 1, linestyle = '--')
plt.show()